# Ordinary Least-Squares Regression for Randomized Control Trials Example

In [25]:
from cais.agent import CausalAgent
from IPython.display import Markdown
import pandas as pd

info_dir = "../../CauSciBench/data/real_info.csv"
data_dir = "../../CauSciBench/data/real_data/"

info = pd.read_csv(info_dir, encoding='latin-1')
selected = info.iloc[6]

data_file = selected['data_files']
description = selected['description']
ground_truth = selected['answer']
query = selected['natural_language_query']

print(f"{query = }")
print(f"{data_file = }")
print(f"description = ")
Markdown(description)

query = 'Does being an immigrant make it less likely to get an interview request?'
data_file = 'vernby_2019.csv'
description = 


The data comes from a randomized field experiment designed to assess whether a candidate's background influences their likelihood of receiving a job interview or offer. Researchers submitted fictitious job applications to restaurants and cafes across Sweden at random. The applications varied in terms of country of birth, gender, citizenship status, work experience, and religious activity. A positive response was defined as a job offer, interview invitation, or follow-up inquiry, while a negative response included any other reply or no response at all. Variables: name: Name of the candidate; stad: City; citizen: 1 if the candidate is a Swedish citizen, 0 otherwise; religious: 1 if the candidate is religious, 0 otherwise; experience: 1 if the candidate has work experience, 0 otherwise; poland: 1 if the candidate was born in Poland, 0 otherwise; iraq: 1 if the candidate was born in Iraq, 0 otherwise; somalia: 1 if the candidate was born in Somalia, 0 otherwise; skilledjob: 1 if the job is high-skilled, 0 otherwise; woman: 1 if the candidate is a woman, 0 otherwise; invited: 1 if the candidate received an interview or a job or a follow-up response, 0 otherwise; city1, city2, city3, city4, city5, city6, city7: Dummy variables for the seven cities; immigrant: 1 if the candidate is an immigrant (not born in Sweden), 0 otherwise; time: Proportion of the applicant's life spent living in Sweden (scaled between 0 and 1)

In [26]:
agent = CausalAgent(
    dataset_path = data_dir + data_file,
    dataset_description = description,
    model_name = 'gpt-4o',
    provider = 'openai'
) # construct agent

In [27]:
agent.analyse_dataset(
    query = query
)

LLM did not identify any temporal or unit variables


Interpreting query with hybrid approach...
Identified Confounders: ['citizen', 'religious', 'experience', 'skilledjob', 'woman', 'time']


LLM reference level '0' not in sampled values for 'immigrant'.


In [28]:
agent.select_method(
    query = query,
    llm_decision=False
)

'regression_discontinuity_design'

In [29]:
agent.select_method(
    query = query,
    llm_decision=True
)

'linear_regression'

In [30]:
agent.select_controls() # can pass a query or use the most recently used one

Using LLM to refine covariate list for controls selection
Method Name:  linear_regression


LLM suggested controls not found in initial usable list.


LLM parsed result: covariates=['citizen', 'religious', 'experience', 'skilledjob', 'woman', 'city1', 'city2', 'city3', 'city4', 'city5', 'city6', 'city7', 'time'] reasoning=None
LLM refined controls to: ['citizen', 'religious', 'experience', 'skilledjob', 'woman', 'city1', 'city2', 'city3', 'city4', 'city5', 'city6', 'city7']


In [31]:
agent.clean_dataset() # generates code to clean the dataset, subject to strict guidelines

'/Users/tae/Work/zhijing/CauSciBench/data/real_data/vernby_2019_cleaned_55317.csv'

In [32]:
agent.execute_method()

effect_estimates_by_level: {'treatment_effect': {'estimate': np.float64(-0.08611704628931967), 'p_value': np.float64(0.0005900321757260882), 'conf_int': [-0.13517178151694775, -0.037062311061691576], 'std_err': np.float64(0.02500797293228001)}}
LLM interpretation raw output: The analysis indicates that being an immigrant is associated with a decrease in the likelihood of receiving an interview request, with an estimated effect of -0.086 (SE = 0.025), and a 95% confidence interval of [-0.135, -0.037]. The p-value of 0.00059 suggests that this result is statistically significant (p < 0.05). The use of linear regression is plausible given the randomized controlled trial design, which helps control for confounding variables, making it more suitable than alternative methods that may not adequately address potential biases. However, key threats to identification validity include the assumptions of linearity and the potential for unmeasured confounders, despite the RCT design. Limitations inc

{'effect_estimate': np.float64(-0.08611704628931967),
 'p_value': np.float64(0.0005900321757260882),
 'confidence_interval': [-0.13517178151694775, -0.037062311061691576],
 'standard_error': np.float64(0.02500797293228001),
 'estimated_effects_by_level': None,
 'reference_level_used': None,
 'formula': 'invited ~ immigrant + citizen + religious + experience + skilledjob + woman + time',
 'model_summary_text': '                            OLS Regression Results                            \n==============================================================================\nDep. Variable:                invited   R-squared:                       0.069\nModel:                            OLS   Adj. R-squared:                  0.065\nMethod:                 Least Squares   F-statistic:                     15.83\nDate:                Tue, 17 Mar 2026   Prob (F-statistic):           3.72e-20\nTime:                        15:21:40   Log-Likelihood:                -396.79\nNo. Observations:         

In [34]:
ground_truth

In [35]:
Markdown(agent.explanations['final_explanation_text'])

**Method Used:** Linear Regression

**Method Explanation:**
The linear_regression method is a causal inference technique used to estimate causal effects from observational data.

**Results:**
- Estimated Causal Effect: -0.0861
- 95% Confidence Interval: [-0.1352, -0.0371]
- P-value: 0.0006

**Interpretation Guide:**
The estimated effect represents the causal impact of immigrant on invited, given the assumptions of the method are met. Careful consideration of these assumptions is needed for valid causal interpretation.

**Assumptions:**
- linear relationship between treatment, covariates, and outcome: This is a key assumption for the selected causal inference method.
- no unmeasured confounders (if observational): This is a key assumption for the selected causal inference method.
- correct model specification: This is a key assumption for the selected causal inference method.
- homoscedasticity of errors: This is a key assumption for the selected causal inference method.
- normally distributed errors (for inference): This is a key assumption for the selected causal inference method.

**Limitations:**
The linear_regression method has general limitations in terms of its assumptions and applicability.



In [36]:
Markdown(agent.explanations['interpretation_text'])

The analysis indicates that being an immigrant is associated with a decrease in the likelihood of receiving an interview request, with an estimated effect of -0.086 (SE = 0.025), and a 95% confidence interval of [-0.135, -0.037]. The p-value of 0.00059 suggests that this result is statistically significant (p < 0.05). The use of linear regression is plausible given the randomized controlled trial design, which helps control for confounding variables, making it more suitable than alternative methods that may not adequately address potential biases. However, key threats to identification validity include the assumptions of linearity and the potential for unmeasured confounders, despite the RCT design. Limitations include the generalizability of the findings beyond the specific context of the study and the possibility that other unobserved factors may still influence the outcome.

In [33]:
#agent.run_analysis(query=query)